# Research Discussion Assignment 2

**Course:** DATA 612 – Recommender Systems  
**Student:** Zoran Glisovic  
**Date:** 2026-06-08

**Video:** [Music Recommendations at Scale with Spark – Christopher Johnson (Spotify)](https://www.youtube.com/watch?v=3LBgiFch4_g)

I worked at a radio station back in the 90s, and one of the things we did was curate playlists. Part of it was popular demand — requests, charts, what was getting airplay everywhere else. But honestly a good chunk was just a personal taste. You'd throw in a song because it felt right for the hour, or because you had a hunch people would respond to it. There was no data. It was vibes, experience, and the occasional angry call when you played something nobody wanted at 8am.

Watching Christopher Johnson's talk brought that back, because the first thing he does is describe exactly that system — manual curation — and say it doesn't scale. What I found interesting is that Spotify's implicit feedback model is trying to reconstruct what a good DJ was doing intuitively. Not asking people what they like, but watching what they do. Play counts instead of request calls.

Confidence weighting took me a bit to wrap my head around. The problem is you can't just treat play counts as ratings — someone leaving Spotify running in the background while they cook dinner looks identical in the data to someone who deliberately played a song five times. The way Hu, Koren and Volinsky handle it is to keep the whole matrix binary (played it / didn't play it) and then separately multiply in a confidence score based on how many times that actually happened. So the 500-play track carries a lot more weight in the loss function than the one-play track. They're capturing two different things — whether an interaction occurred, and how much to actually trust it.

The mathematical engine behind this is ALS (Alternating Least Squares). The ratings matrix is approximated as the product of two smaller matrices, one for users and one for songs, each with approximately 100 latent dimensions. ALS solves them by alternating: fix the song vectors, solve for the user vectors, then flip it and repeat. The confidence weights from the implicit model are directly used as observation weights in that regression.

What the talk doesn't address is the other side of the radio analogy. If you only ever played what was most requested, you'd end up with the same 40 songs on rotation. Listeners would tune out. Spotify's CF system has the same problem — it reinforces what you already know. The cold start problem (new songs nobody has played yet) and the filter bubble are the implicit feedback model's real blind spots, and layering in audio embeddings or NLP is essentially the algorithmic version of a DJ actually listening to new music instead of just reading the request list.

On the Spark side, loading the ratings matrix into memory and caching it across iterations is the obvious fix for Hadoop's disk I/O bottleneck, and the half-gridify method (which MLlib now uses) eliminates the extra shuffle phase by keeping a user's ratings on one partition. The most honest moment of the talk is the ending — they still couldn't run on their full dataset as of 2014, only about 20%. A system that works on 20% of your data is a prototype, not a production recommender.